# 04 — Side-by-side analysis, expert specialisation and verdict

Pulls together the runs from notebooks 01–03 (read from `results/*.json`; the same data lives in Aim).

* all runs in one table and one figure,
* what the experts actually learned to do (which tokens go where),
* the checklist that answers the assignment.

In [ ]:
import sys, os, math, time, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.environ.get("MOE_SRC", os.path.abspath("../src")))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from moe.config import get_config
from moe.utils import set_seed, get_device, savefig, plot_style, aim_repo, work_root, ckpt_dir, results_dir, is_smoke
from moe.data import load_dataset_tokens
from moe.model import GPT, GPTConfig, MoEMLP, param_report
from moe.runs import dense_config, moe_config, train_config, flops_per_token
from moe.trainer import train, load_history, load_model, lr_at
from moe.upcycle import upcycle, max_logit_diff
from moe.viz import ema, color, heatmap_load
from moe.analysis import make_decoder, make_encoder

plot_style()
X = get_config()
DEVICE = get_device()
torch.set_float32_matmul_precision("high")
set_seed(X.seed)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

print(f"torch {torch.__version__} | device = {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)" if DEVICE == "cuda" else ""))
print(f"preset = {X.name!r}   (override with MOE_PRESET=smoke|small|base)")
print(f"Aim repo : {aim_repo()}   -> browse with:  uv run aim up")
print()
print(pd.Series(X.to_dict(), name="value").to_string())

## 1. All runs

In [ ]:
labels = ["dense", "moe_scratch", "dense_continue", "moe_upcycled"]
H = {}
for l in labels:
    try: H[l] = load_history(l)
    except FileNotFoundError: print(f"missing run '{l}' - run the earlier notebooks")
tok_step = X.batch_size * X.block_size
rows = {}
for l, h in H.items():
    rows[l] = {"phase": h["phase"], "steps": h["step"][-1] + 1 - h["step_offset"] if h["step"] else 0,
               "tokens (M)": (len(h["step"]) and (h["step"][-1] + 1 - h["step_offset"]) * tok_step / 1e6),
               "params (M)": h["n_params"] / 1e6, "active (M)": h["n_active_params"] / 1e6,
               "final val": h["final_val_loss"], "val ppl": math.exp(h["final_val_loss"]),
               "min/run": h["wall_clock_s"] / 60, "k tok/s": np.mean(h["tok_s"][1:]) / 1e3,
               "peak GB": h["peak_mem_gb"]}
print(pd.DataFrame(rows).T.round(3).to_string())

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(17, 5))
ax = axs[0]
for l, h in H.items():
    ax.plot(h["eval_step"], h["val_loss"], "o-", ms=3, color=color(l), label=l, ls="-" if l != "moe_scratch" else "--")
ax.axvline(X.dense_steps, color="k", ls=":", lw=1); ax.set_xlabel("global step"); ax.set_ylabel("validation loss"); ax.set_title("validation loss, all runs")
ax.legend()
lo = min(min(h["val_loss"]) for h in H.values()); ax.set_ylim(lo - 0.05, lo + 2.0)

ax = axs[1]
for l, h in H.items():
    if l in ("dense_continue", "moe_upcycled", "dense"):
        x = (np.array(h["eval_step"]) * tok_step) / 1e6
        ax.plot(x, h["val_loss"], "o-", ms=3, color=color(l), label=l)
ax.set_xlabel("training tokens (M)"); ax.set_title("same data, tokens on the x-axis"); ax.legend()
ax.set_ylim(lo - 0.05, lo + 1.0)
plt.tight_layout(); savefig(fig, "04_all_runs.png"); plt.show()

## 2. What did the experts learn?

We run the upcycled MoE over held-out text and record each token's top-1 expert. If routing were meaningless
(experts remained clones), every expert would see the same token distribution. *Lift* = P(token | expert) / P(token): values ≫ 1 mean the expert is
specialised on that token.

In [ ]:
from moe.analysis import collect_routing, expert_similarity
path = os.path.join(ckpt_dir(), "moe_upcycled.pt")
assert os.path.exists(path), "run notebook 03 first"
data = load_dataset_tokens(X.dataset)
model = load_model(path, DEVICE)
dec = make_decoder(X.dataset)
nbatch = 4 if is_smoke() else 40
R = collect_routing(model, data, nbatch, X.batch_size, X.block_size, DEVICE)
toks, top1 = R["tokens"], R["top1"]
print(f"routed {len(toks):,} validation tokens through {top1.shape[0]} MoE layers")
E = X.n_experts
base_counts = np.bincount(toks, minlength=X.vocab_size) / len(toks)

for li in range(top1.shape[0]):
    print(f"\n=========== MoE layer {li} (block {R['layers'][li]}) ===========")
    rows = []
    for e in range(E):
        m = top1[li] == e
        n = int(m.sum())
        if n == 0:
            rows.append({"expert": e, "share": 0.0, "top tokens (lift)": "(unused)"}); continue
        c = np.bincount(toks[m], minlength=X.vocab_size) / n
        score = c / np.maximum(base_counts, 1e-12) * (c > 0)
        cand = np.argsort(-c)[:40]                              # frequent in this expert...
        best = sorted(cand, key=lambda t: -score[t])[:6]        # ...ordered by lift
        rows.append({"expert": e, "share": n / len(toks),
                     "top tokens (lift)": "  ".join(f"{dec([t])!r}x{score[t]:.1f}" for t in best)})
    print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
def category(s: str) -> str:
    if s.strip() == "": return "whitespace/newline"
    t = s.strip()
    if t.isdigit(): return "digits"
    if t.isalpha(): return "word-start" if s.startswith(" ") else "word-piece"
    if not any(ch.isalnum() for ch in t): return "punctuation"
    return "mixed/other"

cats = ["word-start", "word-piece", "punctuation", "digits", "whitespace/newline", "mixed/other"]
cat_of = {int(t): category(dec([t])) for t in np.unique(toks)}
cat_arr = np.array([cat_of[int(t)] for t in toks])
fig, axs = plt.subplots(1, top1.shape[0], figsize=(7 * top1.shape[0], 4), squeeze=False)
for li in range(top1.shape[0]):
    ax = axs[0, li]; bottom = np.zeros(E)
    for c in cats:
        frac = np.array([(cat_arr[top1[li] == e] == c).mean() if (top1[li] == e).any() else 0 for e in range(E)])
        ax.bar(range(E), frac, bottom=bottom, label=c); bottom += frac
    ax.set_title(f"MoE layer {li}: token types per expert"); ax.set_xlabel("expert"); ax.set_ylabel("share of the expert's tokens")
    if li == 0: ax.legend(fontsize=7, loc="lower left")
plt.tight_layout(); savefig(fig, "04_expert_token_types.png"); plt.show()
overall = pd.Series(cat_arr).value_counts(normalize=True).round(3)
print("overall token-type mix:", overall.to_dict())

## 3. Verdict

In [ ]:
d, up_, dc = H.get("dense"), H.get("moe_upcycled"), H.get("dense_continue")
ms = H.get("moe_scratch")
print("CHECKLIST")
print("-" * 78)
if d and up_:
    delta0 = up_["val_loss"][0] - d["final_val_loss"]
    print(f"[{'x' if abs(delta0) < 1e-3 else ' '}] conversion is lossless:      MoE loss at conversion {up_['val_loss'][0]:.4f} vs dense {d['final_val_loss']:.4f} (Δ {delta0:+.5f})")
    print(f"[{'x' if up_['final_val_loss'] < up_['val_loss'][0] else ' '}] MoE keeps training:          {up_['val_loss'][0]:.4f} -> {up_['final_val_loss']:.4f} over {X.cont_steps} steps")
    if dc:
        print(f"[{'x' if up_['final_val_loss'] < dc['final_val_loss'] else ' '}] MoE beats dense control:     {up_['final_val_loss']:.4f} vs {dc['final_val_loss']:.4f} ({up_['final_val_loss'] - dc['final_val_loss']:+.4f})")
if up_:
    print(f"[{'x' if up_['dead_experts'][-1] == 0 else ' '}] no dead experts at the end:  {up_['dead_experts'][-1]} dead")
    off = [s[~np.eye(len(s), dtype=bool)].mean() for s in expert_similarity(model)]
    print(f"[{'x' if max(off) < 0.9999 else ' '}] experts diverged from clones: mean cosine sim per layer {np.round(off, 5).tolist()}")
if ms and d:
    print(f"[i] MoE from scratch vs dense (same budget): {ms['final_val_loss']:.4f} vs {d['final_val_loss']:.4f} ({ms['final_val_loss'] - d['final_val_loss']:+.4f})")
print("-" * 78)
print("Browse everything interactively:  uv run aim up   (experiments: session14, session14_ablations)")